In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

### Q1 How many lesson pages are in the dataset? (Ans = 72)

In [6]:
print("Number of pages =",len(documents))

Number of pages = 72


### Q2. Indexing and searching (answer: 01-agentic-rag/lessons/14-agentic-loop.md)

In [7]:
from minsearch import Index

In [8]:
# Index the documents with minsearch - make content a text field and filename a keyword 
# field. Then search with this query:

# How does the agentic loop keep calling the model until it stops?

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(documents)


In [10]:
search_results = index.search('How does the agentic loop keep calling the model until it stops?')
print(search_results[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


### Q3. RAG

In [ ]:

from openai import OpenAI

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

PROMPT_TEMPLATE = """
QUESTION: {question}

CONTEXT:
{context}
""".strip()

## Customizing the RAGBase Class to suit the new needs
class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model="gpt-5.4-mini"
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model


    def search(self, query, num_results=5):
        return self.index.search(
            query,
            num_results=num_results,
        )
    

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc["content"])
            lines.append("")

        return "\n".join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )
    
    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text
    
    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer

In [26]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the course notes.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [29]:
response = assistant.rag('How does the agentic loop keep calling the model until it stops?')
print(response)

The loop keeps calling the model by using a `while True` loop. On each turn:

1. It calls the model with the current `messages`.
2. It checks the response for any `function_call` items.
3. If there are function calls, it runs the tool, appends the tool output to `messages`, and loops again.
4. If there are no function calls, it breaks out of the loop.

So the stopping condition is simple: **no function calls this turn means the agent is done**.


In [40]:
# creating the function answer the que
# Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

# 700
# 7000 (answer)
# 70000
# 700000

def get_tokens(query, index):
    def search(query, num_results=5):
        return index.search(
            query,
            num_results=num_results,
        )
    
    def build_context( search_results):
        lines = []

        for doc in search_results:
            lines.append(doc["content"])
            lines.append("")

        return "\n".join(lines).strip()

    def build_prompt(query, search_results):
        context =build_context(search_results)
        return PROMPT_TEMPLATE.format(
            question=query, context=context
        )
    
    llm_client = OpenAI()
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    input_messages = [
            {"role": "developer", "content": custom_instructions},
            {"role": "user", "content": prompt}
        ]

    response = llm_client.responses.create(
            model= "gpt-5.4-mini",
            input=input_messages
        )

    return response.usage

In [41]:
usage = get_tokens('How does the agentic loop keep calling the model until it stops?', index)
print('input tokens =', usage.input_tokens)

input tokens = 7020


In [42]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

### Q. How many chunks do you get? (Answer = 295)

In [38]:
print("Number of chunks =", len(chunks))

Number of chunks = 295


### Q5. RAG with chunking (answer ~ 3x fewer (2205 against 7020))

In [39]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

In [43]:
usage_with_chunks = get_tokens('How does the agentic loop keep calling the model until it stops?', chunk_index)
print('input tokens =', usage_with_chunks.input_tokens)

input tokens = 2205


### Q6. Turning it into an agent

In [44]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [47]:
def search(query: str, num_results: str) -> dict[str, str]:
    """
    Search the course notes for entries matching the given query.
    args:
        query: The query from the user
        num_results: the number of results to be fetched from the index
    """
    return index.search(
        query,
        num_results=num_results,
    )

In [48]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [49]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course notes for entries matching the given query.\nargs:\n    query: The query from the user\n    num_results: the number of results to be fetched from the index',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'},
    'num_results': {'type': 'string', 'description': 'num_results parameter'}},
   'required': ['query', 'num_results'],
   'additionalProperties': False}}]

In [51]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=INSTRUCTIONS,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [52]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


In [53]:
result.cost

CostInfo(input_cost=Decimal('0.000285'), output_cost=Decimal('0.00018'), total_cost=Decimal('0.000465'))

In [54]:
result.all_messages

[EasyInputMessage(content='\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n', role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama run locally install local setup ollama", "num_results":"5"}', call_id='call_JX3fOl8eAzDdkPz7cDmzsxBm', name='search', type='function_call', id='fc_0f22c2c68beaa7c7006a3e518a10848191984849d9fb7531ab', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_JX3fOl8eAzDdkPz7cDmzsxBm',
  'output': '{\n  "error": "UFuncTypeError: ufunc \'less\' did not contain a loop with signature matching types (<class \'numpy.dtypes.Int64DType\'>, <class \'numpy.dtypes.StrDType\'>) -> None"\n}'},
 ResponseOutpu

In [55]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received
